# Feature Engineering for Next-Day PM2.5 Forecasting

## Objective

The objective of this notebook is to transform the cleaned daily air-pollution and meteorological dataset into a modelling-ready dataset for forecasting **next-day PM2.5 concentrations** across five Indian cities.

For each city and date, the forecasting task is defined as:

> Use information available on the current day and from previous days to predict PM2.5 concentration on the following day.

### Feature Engineering Strategy

The following features will be developed:

1. Next-day PM2.5 prediction target
2. Calendar and seasonal features
3. Historical PM2.5 lag features
4. Rolling historical PM2.5 statistics
5. Current-day pollutant and meteorological predictors
6. City information

Special attention is given to preventing **data leakage**. No feature will use information from the future relative to the day on which the forecast is made.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

df = pd.read_csv(
    "air_pollution_weather_cleaned_2020_2021 (1).csv"
)

df["Date"] = pd.to_datetime(df["Date"])

df = (
    df
    .sort_values(["City", "Date"])
    .reset_index(drop=True)
)

print("Dataset shape:", df.shape)
print(
    "Date range:",
    df["Date"].min(),
    "to",
    df["Date"].max()
)

print("\nRows by city:")
print(df["City"].value_counts().sort_index())

display(df.head())

Dataset shape: (3655, 13)
Date range: 2020-01-01 00:00:00 to 2021-12-31 00:00:00

Rows by city:
City
Bengaluru    731
Chennai      731
Delhi        731
Kolkata      731
Mumbai       731
Name: count, dtype: int64


,Date,City,PM2.5,PM10,NO2,SO2,O3,CO,Temperature_Mean,Relative_Humidity_Mean,Precipitation,Wind_Speed_Max,Surface_Pressure_Mean
0,2020-01-01,Bengaluru,26.703125,66.219618,31.506189,6.228033,30.566349,247.754587,21.6,80,1.9,18.5,915.0
1,2020-01-02,Bengaluru,24.315661,62.571429,28.996195,5.079903,21.913704,864.569079,22.4,77,1.0,15.1,916.0
2,2020-01-03,Bengaluru,29.665271,72.838235,29.614459,5.535566,30.127463,668.727273,23.0,73,0.0,14.0,915.0
3,2020-01-04,Bengaluru,55.235786,114.010870,35.472778,6.702294,38.663864,1069.550189,23.4,70,0.0,13.2,913.6
4,2020-01-05,Bengaluru,51.294156,101.563218,30.340575,8.320714,43.419048,688.467578,23.3,73,0.2,16.6,913.4


## 1. Creation of the Forecasting Target

The prediction target is defined as the PM2.5 concentration for the **following calendar day in the same city**.

The target is created separately within each city to ensure that observations from different cities are never mixed.

A calendar-day continuity check is also applied. This is important because some pollution observations are missing: simply shifting the PM2.5 column by one row could incorrectly treat the next available observation several days later as the "next day."

Therefore, the target is retained only when the following observation corresponds to exactly one calendar day after the current observation.

In [3]:
# Identify the next observation within each city

df["Next_Date"] = (
    df.groupby("City")["Date"]
    .shift(-1)
)

# Shift PM2.5 within each city

df["PM2.5_Next_Day"] = (
    df.groupby("City")["PM2.5"]
    .shift(-1)
)

# Verify that the shifted observation is exactly the next calendar day

valid_next_day = (
    df["Next_Date"] - df["Date"]
    == pd.Timedelta(days=1)
)

# Remove targets that do not represent the actual next calendar day

df.loc[
    ~valid_next_day,
    "PM2.5_Next_Day"
] = np.nan

print("Rows:", len(df))

print(
    "\nAvailable next-day targets:",
    df["PM2.5_Next_Day"].notna().sum()
)

print(
    "Missing next-day targets:",
    df["PM2.5_Next_Day"].isna().sum()
)

display(
    df[
        [
            "Date",
            "City",
            "PM2.5",
            "Next_Date",
            "PM2.5_Next_Day"
        ]
    ].head(10)
)

Rows: 3655

Available next-day targets: 3498
Missing next-day targets: 157


,Date,City,PM2.5,Next_Date,PM2.5_Next_Day
0,2020-01-01,Bengaluru,26.703125,2020-01-02,24.315661
1,2020-01-02,Bengaluru,24.315661,2020-01-03,29.665271
2,2020-01-03,Bengaluru,29.665271,2020-01-04,55.235786
3,2020-01-04,Bengaluru,55.235786,2020-01-05,51.294156
4,2020-01-05,Bengaluru,51.294156,2020-01-06,22.642570
5,2020-01-06,Bengaluru,22.642570,2020-01-07,24.838682
6,2020-01-07,Bengaluru,24.838682,2020-01-08,31.501083
7,2020-01-08,Bengaluru,31.501083,2020-01-09,41.306190
8,2020-01-09,Bengaluru,41.306190,2020-01-10,38.605655
9,2020-01-10,Bengaluru,38.605655,2020-01-11,31.711675


## 2. Calendar and Seasonal Features

Air pollution exhibits strong temporal and seasonal patterns. Calendar-based variables are therefore created from the forecast-origin date to help the models capture recurring temporal variation.

The following features are generated:

- Year
- Month
- Day of week
- Season

These variables are derived only from the current date and therefore do not introduce future information into the forecasting process.

In [4]:
# Calendar features from the current day's date

df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day_of_Week"] = df["Date"].dt.dayofweek

# Define Indian seasonal categories
def assign_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Pre-Monsoon"
    elif month in [6, 7, 8, 9]:
        return "Monsoon"
    else:
        return "Post-Monsoon"

df["Season"] = df["Month"].apply(assign_season)

print("Season counts:")
print(df["Season"].value_counts())

print("\nCalendar feature preview:")

display(
    df[
        [
            "Date",
            "City",
            "Year",
            "Month",
            "Day_of_Week",
            "Season",
            "PM2.5",
            "PM2.5_Next_Day"
        ]
    ].head(10)
)

Season counts:
Season
Monsoon         1220
Pre-Monsoon      920
Winter           905
Post-Monsoon     610
Name: count, dtype: int64

Calendar feature preview:


,Date,City,Year,Month,Day_of_Week,Season,PM2.5,PM2.5_Next_Day
0,2020-01-01,Bengaluru,2020,1,2,Winter,26.703125,24.315661
1,2020-01-02,Bengaluru,2020,1,3,Winter,24.315661,29.665271
2,2020-01-03,Bengaluru,2020,1,4,Winter,29.665271,55.235786
3,2020-01-04,Bengaluru,2020,1,5,Winter,55.235786,51.294156
4,2020-01-05,Bengaluru,2020,1,6,Winter,51.294156,22.642570
5,2020-01-06,Bengaluru,2020,1,0,Winter,22.642570,24.838682
6,2020-01-07,Bengaluru,2020,1,1,Winter,24.838682,31.501083
7,2020-01-08,Bengaluru,2020,1,2,Winter,31.501083,41.306190
8,2020-01-09,Bengaluru,2020,1,3,Winter,41.306190,38.605655
9,2020-01-10,Bengaluru,2020,1,4,Winter,38.605655,31.711675


## 3. Historical PM2.5 Lag Features

Lagged PM2.5 concentrations are created to capture the temporal persistence of air pollution.

The following historical features are generated:

- PM2.5 concentration 1 day earlier
- PM2.5 concentration 2 days earlier
- PM2.5 concentration 3 days earlier
- PM2.5 concentration 7 days earlier

Lag values are matched using exact calendar dates within each city rather than simple row-based shifting. This ensures that missing dates do not cause observations from incorrect time intervals to be treated as valid lagged measurements.

All lagged variables contain only information from dates prior to the forecast-origin date and therefore do not introduce future-data leakage.

In [5]:
# Create exact calendar-date PM2.5 lookup table

pm25_lookup = df[
    ["City", "Date", "PM2.5"]
].copy()

# Create exact lag features
for lag in [1, 2, 3, 7]:

    lag_lookup = pm25_lookup.copy()

    # Move historical dates forward so they match
    # the date for which they act as a lag
    lag_lookup["Date"] = (
        lag_lookup["Date"]
        + pd.Timedelta(days=lag)
    )

    lag_lookup = lag_lookup.rename(
        columns={
            "PM2.5": f"PM2.5_Lag{lag}"
        }
    )

    df = df.merge(
        lag_lookup,
        on=["City", "Date"],
        how="left"
    )

# Check missing lag values

lag_columns = [
    "PM2.5_Lag1",
    "PM2.5_Lag2",
    "PM2.5_Lag3",
    "PM2.5_Lag7"
]

print("Missing values in lag features:")
print(df[lag_columns].isna().sum())

print("\nLag feature preview:")

display(
    df[
        [
            "Date",
            "City",
            "PM2.5",
            "PM2.5_Lag1",
            "PM2.5_Lag2",
            "PM2.5_Lag3",
            "PM2.5_Lag7",
            "PM2.5_Next_Day"
        ]
    ].head(12)
)

Missing values in lag features:
PM2.5_Lag1    157
PM2.5_Lag2    162
PM2.5_Lag3    167
PM2.5_Lag7    187
dtype: int64

Lag feature preview:


,Date,City,PM2.5,PM2.5_Lag1,PM2.5_Lag2,PM2.5_Lag3,PM2.5_Lag7,PM2.5_Next_Day
0,2020-01-01,Bengaluru,26.703125,NaN,NaN,NaN,NaN,24.315661
1,2020-01-02,Bengaluru,24.315661,26.703125,NaN,NaN,NaN,29.665271
2,2020-01-03,Bengaluru,29.665271,24.315661,26.703125,NaN,NaN,55.235786
3,2020-01-04,Bengaluru,55.235786,29.665271,24.315661,26.703125,NaN,51.294156
4,2020-01-05,Bengaluru,51.294156,55.235786,29.665271,24.315661,NaN,22.642570
5,2020-01-06,Bengaluru,22.642570,51.294156,55.235786,29.665271,NaN,24.838682
6,2020-01-07,Bengaluru,24.838682,22.642570,51.294156,55.235786,NaN,31.501083
7,2020-01-08,Bengaluru,31.501083,24.838682,22.642570,51.294156,26.703125,41.306190
8,2020-01-09,Bengaluru,41.306190,31.501083,24.838682,22.642570,24.315661,38.605655
9,2020-01-10,Bengaluru,38.605655,41.306190,31.501083,24.838682,29.665271,31.711675


## 4. Rolling Historical PM2.5 Features

Rolling historical features are constructed to represent recent PM2.5 behaviour prior to the next-day forecast.

The following features are generated:

- 3-day rolling mean of PM2.5
- 7-day rolling mean of PM2.5
- 7-day rolling standard deviation of PM2.5

The rolling windows use calendar time rather than a fixed number of rows. This ensures that missing dates do not incorrectly expand the historical period represented by a rolling feature.

Because the forecasting task predicts PM2.5 for the following day, measurements available on the current day may legitimately be included in these features. No observations occurring after the forecast-origin date are used.

In [6]:
# Ensure chronological order
df = df.sort_values(
    ["City", "Date"]
).reset_index(drop=True)

# Calculate time-based rolling features separately for each city
rolling_features = []

for city, group in df.groupby("City"):

    group = group.copy()
    group = group.sort_values("Date")

    pm25_series = (
        group
        .set_index("Date")["PM2.5"]
    )

    group["PM2.5_Rolling3_Mean"] = (
        pm25_series
        .rolling("3D", min_periods=2)
        .mean()
        .to_numpy()
    )

    group["PM2.5_Rolling7_Mean"] = (
        pm25_series
        .rolling("7D", min_periods=3)
        .mean()
        .to_numpy()
    )

    group["PM2.5_Rolling7_Std"] = (
        pm25_series
        .rolling("7D", min_periods=3)
        .std()
        .to_numpy()
    )

    rolling_features.append(group)

df = (
    pd.concat(rolling_features, ignore_index=True)
    .sort_values(["City", "Date"])
    .reset_index(drop=True)
)

rolling_columns = [
    "PM2.5_Rolling3_Mean",
    "PM2.5_Rolling7_Mean",
    "PM2.5_Rolling7_Std"
]

print("Missing values in rolling features:")
print(df[rolling_columns].isna().sum())

display(
    df[
        [
            "Date",
            "City",
            "PM2.5",
            "PM2.5_Lag1",
            "PM2.5_Rolling3_Mean",
            "PM2.5_Rolling7_Mean",
            "PM2.5_Rolling7_Std",
            "PM2.5_Next_Day"
        ]
    ].head(12)
)

Missing values in rolling features:
PM2.5_Rolling3_Mean    154
PM2.5_Rolling7_Mean    126
PM2.5_Rolling7_Std     126
dtype: int64


,Date,City,PM2.5,PM2.5_Lag1,PM2.5_Rolling3_Mean,PM2.5_Rolling7_Mean,PM2.5_Rolling7_Std,PM2.5_Next_Day
0,2020-01-01,Bengaluru,26.703125,NaN,NaN,NaN,NaN,24.315661
1,2020-01-02,Bengaluru,24.315661,26.703125,25.509393,NaN,NaN,29.665271
2,2020-01-03,Bengaluru,29.665271,24.315661,26.894686,26.894686,2.679945,55.235786
3,2020-01-04,Bengaluru,55.235786,29.665271,36.405573,33.979961,14.338499,51.294156
4,2020-01-05,Bengaluru,51.294156,55.235786,45.398404,37.442800,14.633888,22.642570
5,2020-01-06,Bengaluru,22.642570,51.294156,43.057504,34.976095,14.416253,24.838682
6,2020-01-07,Bengaluru,24.838682,22.642570,32.925136,33.527893,13.706616,31.501083
7,2020-01-08,Bengaluru,31.501083,24.838682,26.327445,34.213316,13.425534,41.306190
8,2020-01-09,Bengaluru,41.306190,31.501083,32.548652,36.640534,12.861929,38.605655
9,2020-01-10,Bengaluru,38.605655,41.306190,37.137643,37.917732,12.492425,31.711675


## 5. Predictor Definition and Data-Leakage Audit

The forecasting dataset combines information available on the forecast-origin day with historical PM2.5 behaviour.

The predictor groups include:

- Current-day air-pollution measurements
- Current-day meteorological conditions
- Historical PM2.5 lag features
- Rolling historical PM2.5 statistics
- Calendar and seasonal information
- City information

The response variable is `PM2.5_Next_Day`, representing PM2.5 on the following calendar day.

Before constructing the final modelling dataset, the predictors are explicitly reviewed to ensure that no feature contains information from the future relative to the forecast-origin date.

In [7]:
# Define predictor groups

current_pollution_features = [
    "PM2.5",
    "PM10",
    "NO2",
    "SO2",
    "O3",
    "CO"
]

weather_features = [
    "Temperature_Mean",
    "Relative_Humidity_Mean",
    "Precipitation",
    "Wind_Speed_Max",
    "Surface_Pressure_Mean"
]

lag_features = [
    "PM2.5_Lag1",
    "PM2.5_Lag2",
    "PM2.5_Lag3",
    "PM2.5_Lag7"
]

rolling_features = [
    "PM2.5_Rolling3_Mean",
    "PM2.5_Rolling7_Mean",
    "PM2.5_Rolling7_Std"
]

calendar_features = [
    "Year",
    "Month",
    "Day_of_Week",
    "Season"
]

categorical_features = [
    "City",
    "Season"
]

target = "PM2.5_Next_Day"

all_predictors = (
    current_pollution_features
    + weather_features
    + lag_features
    + rolling_features
    + calendar_features
    + ["City"]
)

# Remove duplicates while preserving order
all_predictors = list(dict.fromkeys(all_predictors))

print("Number of predictors:", len(all_predictors))

print("\nPredictors:")
for feature in all_predictors:
    print("-", feature)

print("\nTarget:")
print("-", target)

# Leakage checks
assert target not in all_predictors
assert "Next_Date" not in all_predictors

print("\nLeakage audit passed:")
print("Target and future-date helper variables are not predictors.")

Number of predictors: 23

Predictors:
- PM2.5
- PM10
- NO2
- SO2
- O3
- CO
- Temperature_Mean
- Relative_Humidity_Mean
- Precipitation
- Wind_Speed_Max
- Surface_Pressure_Mean
- PM2.5_Lag1
- PM2.5_Lag2
- PM2.5_Lag3
- PM2.5_Lag7
- PM2.5_Rolling3_Mean
- PM2.5_Rolling7_Mean
- PM2.5_Rolling7_Std
- Year
- Month
- Day_of_Week
- Season
- City

Target:
- PM2.5_Next_Day

Leakage audit passed:
Target and future-date helper variables are not predictors.


In [8]:
required_columns = all_predictors + [target]

missing_model_features = (
    df[required_columns]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print("Missing values before final model-data filtering:")
display(
    missing_model_features[
        missing_model_features > 0
    ].to_frame("Missing_Count")
)

complete_rows = df[required_columns].notna().all(axis=1)

print(
    "\nTotal rows:",
    len(df)
)

print(
    "Rows with complete predictors and target:",
    complete_rows.sum()
)

print(
    "Rows that would be removed:",
    (~complete_rows).sum()
)

print(
    "Percentage retained:",
    round(complete_rows.mean() * 100, 2),
    "%"
)

Missing values before final model-data filtering:


,Missing_Count
PM10,331
SO2,276
NO2,275
O3,274
CO,274
PM2.5_Lag7,187
PM2.5_Lag3,167
PM2.5_Lag2,162
PM2.5_Next_Day,157
PM2.5_Lag1,157



Total rows: 3655
Rows with complete predictors and target: 3207
Rows that would be removed: 448
Percentage retained: 87.74 %


## 6. Construction of the Final Forecasting Dataset

For model development, observations with incomplete values in the selected predictor variables or next-day PM2.5 target are excluded.

The final dataset retains only observations for which the complete predictor information required by the forecasting models is available.

This complete-case dataset contains current-day pollution and meteorological measurements, historical PM2.5 lag and rolling features, temporal variables, city information, and the next-day PM2.5 forecasting target.

In [9]:
# Create final modelling dataset

model_data = (
    df[
        ["Date"]
        + all_predictors
        + [target]
    ]
    .dropna()
    .sort_values(["Date", "City"])
    .reset_index(drop=True)
)

print("Final modelling dataset shape:")
print(model_data.shape)

print("\nDate range:")
print(
    model_data["Date"].min(),
    "to",
    model_data["Date"].max()
)

print("\nRows by city:")
print(
    model_data["City"]
    .value_counts()
    .sort_index()
)

print("\nMissing values remaining:")
print(model_data.isna().sum().sum())

display(model_data.head())

Final modelling dataset shape:
(3207, 25)

Date range:
2020-01-08 00:00:00 to 2021-12-30 00:00:00

Rows by city:
City
Bengaluru    535
Chennai      673
Delhi        677
Kolkata      645
Mumbai       677
Name: count, dtype: int64

Missing values remaining:
0


,Date,PM2.5,PM10,NO2,SO2,O3,CO,Temperature_Mean,Relative_Humidity_Mean,Precipitation,Wind_Speed_Max,Surface_Pressure_Mean,PM2.5_Lag1,PM2.5_Lag2,PM2.5_Lag3,PM2.5_Lag7,PM2.5_Rolling3_Mean,PM2.5_Rolling7_Mean,PM2.5_Rolling7_Std,Year,Month,Day_of_Week,Season,City,PM2.5_Next_Day
0,2020-01-08,31.501083,79.282609,33.996337,4.558495,46.321798,800.207985,22.5,67,0.0,16.4,914.1,24.838682,22.642570,51.294156,26.703125,26.327445,34.213316,13.425534,2020,1,2,Winter,Bengaluru,41.306190
1,2020-01-08,37.818907,61.834590,6.358947,27.412778,33.906346,547.894737,25.4,75,0.1,14.7,1012.2,29.295960,15.925365,13.734690,20.262949,27.680077,23.256471,8.462275,2020,1,2,Winter,Chennai,32.770514
2,2020-01-08,122.183095,148.136364,34.848894,9.403614,11.159524,1120.674043,12.8,92,17.5,14.5,988.9,126.446154,155.419504,173.386555,349.875000,134.682918,189.191787,68.025340,2020,1,2,Winter,Delhi,97.896274
3,2020-01-08,122.455102,220.020879,45.102000,10.188000,26.500769,1255.102041,17.7,80,0.0,8.9,1013.9,115.196364,102.015116,93.950549,121.902564,113.222194,93.766368,34.834783,2020,1,2,Winter,Kolkata,89.153933
4,2020-01-08,68.270000,104.341892,34.320862,8.277315,38.708889,866.755245,23.8,76,0.0,17.3,1012.4,94.860675,71.676000,97.626333,110.954359,78.268892,92.058904,19.772728,2020,1,2,Winter,Mumbai,62.421134


In [10]:
# Final integrity checks

print("Duplicate City-Date rows:")
print(
    model_data.duplicated(
        subset=["City", "Date"]
    ).sum()
)

print("\nTarget summary:")
print(
    model_data["PM2.5_Next_Day"]
    .describe()
    .round(2)
)

print("\nChronological range by city:")
display(
    model_data.groupby("City")["Date"]
    .agg(["min", "max", "count"])
)

# Confirm chronological sorting
is_sorted = (
    model_data
    .sort_values(["Date", "City"])
    .reset_index(drop=True)["Date"]
    .equals(model_data["Date"])
)

print("\nChronologically sorted:", is_sorted)

Duplicate City-Date rows:
0

Target summary:
count    3207.00
mean       51.64
std        58.13
min         0.00
25%        18.61
50%        31.38
75%        63.92
max       985.00
Name: PM2.5_Next_Day, dtype: float64

Chronological range by city:


,min,max,count
City,,,
Bengaluru,2020-01-08,2021-12-30,535
Chennai,2020-01-08,2021-12-30,673
Delhi,2020-01-08,2021-12-30,677
Kolkata,2020-01-08,2021-12-30,645
Mumbai,2020-01-08,2021-12-30,677



Chronologically sorted: True


In [11]:
output_file = "pm25_next_day_forecasting_features.csv"

model_data.to_csv(
    output_file,
    index=False
)

print(
    "Saved:",
    output_file
)

print(
    "Shape:",
    model_data.shape
)

Saved: pm25_next_day_forecasting_features.csv
Shape: (3207, 25)


## 7. Feature Engineering Summary

A modelling-ready dataset was constructed for forecasting next-day PM2.5 concentrations across five Indian cities.

The forecasting target represents PM2.5 concentration on the following calendar day within the same city. Exact calendar-date matching was used when constructing lag features to prevent missing dates from producing incorrectly aligned historical observations.

The final predictor set incorporates current-day pollutant and meteorological conditions, historical PM2.5 lag features, rolling PM2.5 statistics, temporal variables, seasonal information, and city identity.

After excluding observations with incomplete predictor or target information, the final dataset contains **3,207 observations and 25 variables**, covering January 2020 through December 2021. No missing values or duplicate City-Date observations remain.

The resulting dataset is chronologically structured and will be used for time-aware development and evaluation of next-day PM2.5 forecasting models.